# SQL Practice — Window Functions & CTEs

**Format:** 10 classic interview problems (LeetCode / StrataScratch style), paraphrased in my own words.
Engine is **DuckDB in-memory** — no Snowflake, so this burns **zero trial credits** and stays light on RAM.
The SQL you write here is standard ANSI window-function / CTE syntax that runs the same on Snowflake.

**The drill:** ~30 min, fluent pace. Aim ~3 min/problem.

**Rules (the whole point — same as the pandas notebook):**
1. Write every query **yourself**. No AI, no copy-paste.
2. Stuck? Use the **hint**, or read the DuckDB / Postgres window-function docs — not a chatbot.
3. Run your query, compare to the **Expected output** shown under each problem.
4. After each: say out loud *"could I explain this in an interview?"* If not, you're not done.

**Concepts covered:** `ROW_NUMBER` / `RANK` / `DENSE_RANK`, `LAG` / `LEAD`, `PARTITION BY`,
window frames (`ROWS BETWEEN ...`), running totals, moving averages, gaps-and-islands, multi-step CTEs.


## Setup — run this once
Creates all the sample tables and a tiny `run()` helper. Just run it; don't edit it.

In [ ]:
import duckdb
con = duckdb.connect()   # in-memory; nothing written to disk

con.execute("""
CREATE OR REPLACE TABLE Scores(id INTEGER, score DECIMAL(3,2));
INSERT INTO Scores VALUES (1,3.50),(2,3.65),(3,4.00),(4,3.85),(5,4.00),(6,3.65);

CREATE OR REPLACE TABLE Employee(id INTEGER, salary INTEGER);
INSERT INTO Employee VALUES (1,100),(2,200),(3,300);

CREATE OR REPLACE TABLE Emp(id INTEGER, name VARCHAR, salary INTEGER, departmentId INTEGER);
INSERT INTO Emp VALUES
 (1,'Joe',85000,1),(2,'Henry',80000,2),(3,'Sam',60000,2),
 (4,'Max',90000,1),(5,'Janet',69000,1),(6,'Randy',85000,1),(7,'Will',70000,1);
CREATE OR REPLACE TABLE Department(id INTEGER, name VARCHAR);
INSERT INTO Department VALUES (1,'IT'),(2,'Sales');

CREATE OR REPLACE TABLE Logs(id INTEGER, num INTEGER);
INSERT INTO Logs VALUES (1,1),(2,1),(3,1),(4,2),(5,1),(6,2),(7,2);

CREATE OR REPLACE TABLE Weather(id INTEGER, recordDate DATE, temperature INTEGER);
INSERT INTO Weather VALUES (1,'2015-01-01',10),(2,'2015-01-02',25),(3,'2015-01-03',20),(4,'2015-01-04',30);

CREATE OR REPLACE TABLE Customer(customer_id INTEGER, name VARCHAR, visited_on DATE, amount INTEGER);
INSERT INTO Customer VALUES
 (1,'Jhon','2019-01-01',100),(2,'Daniel','2019-01-02',110),(3,'Jade','2019-01-03',120),
 (4,'Khaled','2019-01-04',130),(5,'Winston','2019-01-05',110),(6,'Elvis','2019-01-06',140),
 (7,'Anna','2019-01-07',150),(8,'Maria','2019-01-08',80),(9,'Jaze','2019-01-09',110),
 (1,'Jhon','2019-01-10',130),(3,'Jade','2019-01-10',150);

CREATE OR REPLACE TABLE Sales(sale_date DATE, revenue INTEGER);
INSERT INTO Sales VALUES
 ('2019-01-01',100),('2019-01-02',150),('2019-01-03',120),('2019-01-04',200),('2019-01-05',90);

CREATE OR REPLACE TABLE EmpCo(id INTEGER, company VARCHAR, salary INTEGER);
INSERT INTO EmpCo VALUES
 (1,'A',2341),(2,'A',341),(3,'A',15),(4,'A',15314),(5,'A',451),(6,'A',513),
 (7,'B',15),(8,'B',13),(9,'B',1154),(10,'B',1345),(11,'B',1221),(12,'B',234),
 (13,'C',2345),(14,'C',2645),(15,'C',2645),(16,'C',2652),(17,'C',65);

CREATE OR REPLACE TABLE Stadium(id INTEGER, visit_date DATE, people INTEGER);
INSERT INTO Stadium VALUES
 (1,'2017-01-01',10),(2,'2017-01-02',109),(3,'2017-01-03',150),(4,'2017-01-04',99),
 (5,'2017-01-05',145),(6,'2017-01-06',1455),(7,'2017-01-07',199),(8,'2017-01-09',188);

CREATE OR REPLACE TABLE MonthSales(month_id INTEGER, revenue INTEGER);
INSERT INTO MonthSales VALUES (1,100),(2,150),(3,120),(4,240),(5,300);
""")

def run(sql):
    return con.execute(sql).df()

print("tables ready:",
      [r[0] for r in con.execute("SHOW TABLES").fetchall()])


---
## 1. Rank the scores (DENSE_RANK)
_Based on LeetCode 178 “Rank Scores”._

**Table `Scores(id, score)`** — exam scores.

Return every `score` (highest first) with a **rank** column. Ranking rules:
- highest score = rank 1,
- **ties share the same rank**, and
- ranks have **no gaps** (the score after a tie continues sequentially, not skipping).

**Hint:** that exact "ties tie, no gaps" behaviour is one specific ranking window function — pick it over `RANK`/`ROW_NUMBER` and `ORDER BY score DESC` inside the `OVER()`.

**Expected output (6 rows):**
```
 score  rnk
  4.00    1
  4.00    1
  3.85    2
  3.65    3
  3.65    3
  3.50    4
```

In [ ]:
run("""
-- your query here

""")

---
## 2. Second highest salary
_Based on LeetCode 176 “Second Highest Salary”._

**Table `Employee(id, salary)`.**

Return the **second-highest distinct** salary as `SecondHighestSalary`. If there is no second one, return `NULL` (not an empty result).

**Hint:** two classic ways — (a) `MAX(salary)` of everyone below the overall `MAX`, or (b) `DENSE_RANK` in a subquery then filter `= 2`. Either way wrap it so a missing value comes back as `NULL`. Try it both ways.

**Expected output (1 row):**
```
 SecondHighestSalary
                 200
```

In [ ]:
run("""
-- your query here

""")

---
## 3. Top three earners per department
_Based on LeetCode 185 “Department Top Three Salaries”._

**Tables `Emp(id, name, salary, departmentId)` and `Department(id, name)`.**

A person is a "high earner" in their department if there are **fewer than three distinct salaries higher** than theirs in that department. List `Department`, `Employee` (the person's name), and `salary`.

**Hint:** `DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC)`, join to `Department`, keep rank `<= 3`. Note ties at the same salary both count as the same place.

**Expected output (6 rows):**
```
Department Employee  salary
        IT      Max   90000
        IT      Joe   85000
        IT    Randy   85000
        IT     Will   70000
     Sales    Henry   80000
     Sales      Sam   60000
```

In [ ]:
run("""
-- your query here

""")

---
## 4. Consecutive numbers
_Based on LeetCode 180 “Consecutive Numbers”._

**Table `Logs(id, num)`** — `id` is the sequential order.

Find all numbers that appear **three or more times in a row** (consecutively, by `id`). Return the distinct number(s) as `ConsecutiveNums`.

**Hint:** `LEAD(num,1)` and `LEAD(num,2)` over `ORDER BY id`; keep rows where the current value equals both leads. (Or self-join three copies — but practise the window version.)

**Expected output (1 row):**
```
 ConsecutiveNums
               1
```

In [ ]:
run("""
-- your query here

""")

---
## 5. Rising temperature
_Based on LeetCode 197 “Rising Temperature”._

**Table `Weather(id, recordDate, temperature)`.**

Return the `id` of every day whose temperature was **higher than the previous (calendar) day's**.

**Hint:** `LAG(temperature) OVER (ORDER BY recordDate)` to pull yesterday onto today's row, then compare. (Careful: order by the *date*, not the id, in the general case.)

**Expected output (2 rows):**
```
 id
  2
  4
```

In [ ]:
run("""
-- your query here

""")

---
## 6. Restaurant growth — 7-day moving average
_Based on LeetCode 1321 “Restaurant Growth”._

**Table `Customer(customer_id, name, visited_on, amount)`** — one row per customer visit; multiple visits can share a date.

For each `visited_on`, compute the **7-day window ending on that day** (that day plus the 6 before it):
- `amount` = total paid across those 7 days,
- `average_amount` = that total / 7, rounded to 2 decimals.

Only output days that have a **full 7-day window** behind them (i.e. drop the first 6 dates). Order by `visited_on`.

**Hint:** first roll visits up to one row per day with `SUM(amount) GROUP BY visited_on`. Then a windowed `SUM`/`AVG` with frame `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW`. Use `ROW_NUMBER` (or a date offset) to drop the first 6 days.

**Expected output (4 rows):**
```
visited_on  amount  average_amount
2019-01-07   860.0          122.86
2019-01-08   840.0          120.00
2019-01-09   840.0          120.00
2019-01-10  1000.0          142.86
```

In [ ]:
run("""
-- your query here

""")

---
## 7. Running total of daily revenue
_Classic StrataScratch-style cumulative-sum problem._

**Table `Sales(sale_date, revenue)`** — one row per day.

Return each day's `revenue` plus a `running_total`: the cumulative revenue from the first day up to and including that day. Order by date.

**Hint:** `SUM(revenue) OVER (ORDER BY sale_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`. (With `ORDER BY` and no explicit frame the default is `RANGE UNBOUNDED PRECEDING` — know why that can differ from `ROWS` when dates tie.)

**Expected output (5 rows):**
```
 sale_date  revenue  running_total
2019-01-01      100          100.0
2019-01-02      150          250.0
2019-01-03      120          370.0
2019-01-04      200          570.0
2019-01-05       90          660.0
```

In [ ]:
run("""
-- your query here

""")

---
## 8. Median salary per company
_Based on LeetCode 569 “Median Employee Salary”._

**Table `EmpCo(id, company, salary)`.**

Return the rows that make up the **median salary of each company**. (Odd count → one middle row; even count → the two middle rows.)

**Hint:** within each company, `ROW_NUMBER() OVER (PARTITION BY company ORDER BY salary)` gives position `rn`, and `COUNT(*) OVER (PARTITION BY company)` gives the size `c`. The median rows satisfy `rn BETWEEN c/2.0 AND c/2.0 + 1` — work out the exact integer condition (`rn IN (FLOOR((c+1)/2), FLOOR((c+2)/2))` is one clean form).

**Expected output (5 rows):**
```
 id company  salary
  5       A     451
  6       A     513
 12       B     234
  9       B    1154
 14       C    2645
```

In [ ]:
run("""
-- your query here

""")

---
## 9. Human traffic of the stadium (gaps & islands)
_Based on LeetCode 601 “Human Traffic of Stadium”._

**Table `Stadium(id, visit_date, people)`** — `id` is sequential, `people` is the headcount.

Find every record that is part of a run of **three or more consecutive `id`s** where **each** day had `people >= 100`. Order by `visit_date`.

**Hint:** the gaps-and-islands trick. Filter to `people >= 100`, then `id - ROW_NUMBER() OVER (ORDER BY id)` is **constant within a consecutive run** — group by that key, keep groups with `COUNT(*) >= 3`, then return their rows.

**Expected output (4 rows):**
```
 id visit_date  people
  5 2017-01-05     145
  6 2017-01-06    1455
  7 2017-01-07     199
  8 2017-01-09     188
```

In [ ]:
run("""
-- your query here

""")

---
## 10. Month-over-month growth %
_StrataScratch-style period-over-period growth with `LAG`._

**Table `MonthSales(month_id, revenue)`** — one row per month.

For each month return `revenue` and `pct_growth`: the percent change vs the **previous month**, rounded to 1 decimal. The first month has no prior month, so its growth is `NULL`.

**Hint:** `LAG(revenue) OVER (ORDER BY month_id)` for the prior value, then `100.0 * (revenue - prev) / prev`. (`pandas`/DuckDB will show the first row's null as `NaN`.)

**Expected output (5 rows):**
```
 month_id  revenue  pct_growth
        1      100         NaN
        2      150        50.0
        3      120       -20.0
        4      240       100.0
        5      300        25.0
```

In [ ]:
run("""
-- your query here

""")

---
## Done? Self-review checklist
- Could you **explain each window frame** (`ROWS` vs `RANGE`, `PRECEDING`/`CURRENT ROW`) out loud?
- Do you know **when `RANK` vs `DENSE_RANK` vs `ROW_NUMBER`** changes the answer?
- Which of these needed a **CTE** purely for readability vs. which genuinely needed the staged result?
- Re-do #6 and #9 from scratch tomorrow — they're the ones that show up most in interviews.